In [106]:
import pandas as pd
import numpy as np
import random

In [107]:
school = 'school_1'

# Define paths
raw_data_folder = '../data/raw_data'
processed_data_folder = '../data/processed_data'
processed_data_folder_school = f'{processed_data_folder}/{school}'
print(f'Processed data folder: {processed_data_folder_school}')

Processed data folder: ../data/processed_data/school_1


In [108]:
# Read in processed data
constraints_students = pd.read_csv(f'{processed_data_folder_school}/constraints_students.csv')
constraints_teachers = pd.read_csv(f'{processed_data_folder_school}/constraints_teachers.csv')
current_groups = pd.read_csv(f'{processed_data_folder_school}/current_groups.csv')
group_preferences = pd.read_csv(f'{processed_data_folder_school}/group_preferences.csv')
info_students = pd.read_csv(f'{processed_data_folder_school}/info_students.csv')
info_teachers = pd.read_csv(f'{processed_data_folder_school}/info_teachers.csv')
group_preferences

n_students, n_groups, min_group_size, max_extra_care_1, max_extra_care_2 = group_preferences.iloc[0]


In [109]:
def get_max_group_size(min_group_size, n_students, n_groups):
    remaining = n_students - ( min_group_size * n_groups)
    max_size = min_group_size + (remaining // n_groups)
    if remaining % n_groups != 0:
        max_size += 1
    return max_size

max_group_size = get_max_group_size(min_group_size, n_students, n_groups)

In [110]:
# Set seed for reproducibility
seed = 42

teachers_exclude = constraints_teachers[constraints_teachers['Together'] == 'No']
teachers_include = constraints_teachers[constraints_teachers['Together'] == 'Yes']
students_include = constraints_students[constraints_students['Together'] == 'Yes']
students_exclude = constraints_students[constraints_students['Together'] == 'No']

# Initialize groups
groups = {}
for _, teacher in enumerate(info_teachers['Teacher']):
    groups[teacher] = []

In [111]:
def get_assigned_students(groups):
    return [student for group in groups.values() for student in group]

def is_assigned(student, groups):
    if student in get_assigned_students(groups):
        return True
    return False

def get_group(student, groups):
    group = [group for group in groups if student in groups[group]]
    return group[0]


# def list_constraint_students(constraints_students, constraints_teachers):
#     # Create set of all students mentioned in constraints students and teachers
#     students_in_constraints = set(constraints_students['Student 1']).union(set(constraints_students['Student 2']))
#     students_in_constraints.update(set(constraints_teachers['Student']))
#     return students_in_constraints

In [112]:
# First assign students that can only have one option
def assign_includes(groups, constraints_teachers, constraints_students):
    teachers_include = constraints_teachers[constraints_teachers['Together'] == 'Yes']
    students_include = constraints_students[constraints_students['Together'] == 'Yes']

    # Assign students to teachers based on inclusion constraints
    for _, (student, teacher, _) in teachers_include.iterrows():
        groups[teacher].append(student)

    # Add students that must be together to the same group if already assigned
    for _, (s1, s2, _) in students_include.iterrows():
        if (is_assigned(s1, groups) and is_assigned(s2, groups)) or (not is_assigned(s1, groups) and not is_assigned(s2, groups)):
            # If both students are already assigned or not assigned, do nothing
            continue
        elif is_assigned(s1, groups):
            group = get_group(s1, groups)
            groups[group].append(s2)
        elif is_assigned(s2, groups):
            group = get_group(s2, groups)
            groups[group].append(s1)

    return groups

In [113]:
# This function only checks max assignments of a certain binary column - if there are minimum constraints an extra check is needed
def violates_binary(student, group, groups, student_info, col, limit):
   value = student_info.loc[student_info['Student'] == student, col].values[0]
   if value == 'No':
      return False

   # Get number of binary students in the group
   count = student_info[student_info['Student'].isin(groups[group]) & (student_info[col] == 'Yes')].shape[0]

   # Check if the number of binary students in the group exceeds the limit
   if count >= limit:
      return True

   return False

def violates_min_group_size(group, groups, min_group_size):
    if len(groups[group]) < min_group_size:
        return True
    return False

def violates_max_group_size(group, groups, max_group_size):
      if len(groups[group]) > max_group_size:
         return True
      return False


In [114]:
def violates_student_pair(student, group, groups, constraints_students):
    # Check if student is in constraints
    mask = (constraints_students['Student 1'] == student) | (constraints_students['Student 2'] == student)
    if not mask.any():
        return False

    # Get the other students
    matches = constraints_students[mask].copy()
    matches['Other'] = matches.apply(lambda row: row['Student 2'] if row['Student 1'] == student else row['Student 1'], axis=1)

    includes = matches[matches['Together'] == 'Yes']['Other'].tolist()
    excludes = matches[matches['Together'] != 'Yes']['Other'].tolist()

    # Check if all includes are in the group
    for include in includes:
        if include not in groups[group]:
            return True

    # Check if any excludes are in the group
    for exclude in excludes:
        if exclude in groups[group]:
            return True

    return False

In [115]:
def violates_teacher_pair(student, group, constraints_teachers):
    # Check if student teacher pair is in constraint
    match = constraints_teachers[(constraints_teachers['Student'] == student) & (constraints_teachers['Teacher'] == group)]

    if match.empty:
        return False

    # Check if they are allowed to be together
    if match['Together'].values[0] == 'No':
        return True

    return False

In [ ]:
def main(seed, groups):
    # Set random seed for reproducibility
    random.seed(seed)

    # First assign students that can only have one option
    groups = assign_includes(groups, constraints_teachers, constraints_students)
    print(f'Groups after assigning includes: {groups}')

    # TODO: Implement the main assignment logic
    # Randomly shuffle students
    # Then check if they can be assigned to a group


main(seed, groups)

Groups after assigning includes: {'T_01': ['S_28'], 'T_02': [], 'T_03': ['S_10', 'S_02', 'S_30', 'S_29', 'S_13']}
Check violates teacher pair: False
